In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import torchtext
import tqdm
import evaluate

In [ ]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# 数据集 Dataset

## 1、添加一个Markdown单元格，在其中解释下方单元格的两行代码。
设置 os.environ['HF_ENDPOINT'] = \'https://hf-mirror.com' ，这样做具体改变了什么？
为什么要设置HF_ENDPOINT=\'https://hf-mirror.com'而非直接使用官方源？
dataset = datasets.load_dataset("bentrevett/multi30k") 这行代码具体完成了什么操作？

**解释：**

`os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'`：将 Hugging Face 的下载源临时替换为国内镜像站 hf-mirror.com。默认官方源 `https://huggingface.co` 在国内访问速度极慢甚至无法访问，切换到国内镜像可大幅提升模型和数据集的下载速度。

`dataset = datasets.load_dataset("bentrevett/multi30k")`：从 Hugging Face Hub（经由镜像）下载并加载 **Multi30k** 数据集。该数据集包含约 3 万条英语-德语平行句对，常用于机器翻译任务的基准测试。函数返回一个 `DatasetDict` 对象，内含 train/validation/test 三个子集。

In [ ]:
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
dataset = datasets.load_dataset("bentrevett/multi30k")

## 2、运行下方的单元格。
你会看到数据集对象（一个DatasetDict）包含训练、验证和测试集，每个集合中的样本数量，以及每个集合中的特征（“en”和“de”）。


In [ ]:
dataset

In [ ]:
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

## 3、运行下方的单元格。
我们可以索引每个数据集来查看单个示例。每个例子都有两个特征：“en”和“de”，是对应的英语和德语。


In [ ]:
train_data[0]

接下来我们进行分词。英语/德语的分词较中文要直接，比如句子"good morning!会被分词为["good", "morning", "!"]序列。

下方的代码要成功安装en_core_web_sm和de_core_news_sm后才不会报错。

# 分词器 Tokenizers

In [ ]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

## 4、运行下方的单元格。
我们可以使用.tokenizer方法调用每个spaCy模型的分词器，该方法接受字符串并返回Token对象序列。我们可以使用text属性从Token对象中获取字符串。


In [ ]:
string = "What a lovely day it is today!"

[token.text for token in en_nlp.tokenizer(string)]

## 5、添加一个Markdown单元格，在其中解释下方单元格的函数的作用。


**`tokenize_example` 函数的作用：**

该函数对数据集中的单条样本同时进行英语和德语的分词处理，具体步骤如下：

1. 使用 spaCy 的英语/德语分词器分别对 `example["en"]` 和 `example["de"]` 进行分词，得到 Token 文本列表；
2. 通过切片 `[:max_length]` 截断超长序列，防止序列过长导致内存溢出；
3. 若 `lower=True`，则将所有 Token 转换为小写，统一大小写表示；
4. 在序列首尾分别添加起始符 `<sos>` 和终止符 `<eos>`，为后续模型处理标注边界；
5. 返回包含 `en_tokens` 和 `de_tokens` 两个新字段的字典，供 `map` 函数写回数据集。

In [ ]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

## 6、添加一个Markdown单元格，在其中解释下方单元格出现的\<sos>和\<eos>的含义，以及map函数的作用。


**`<sos>` 和 `<eos>` 的含义：**

- `<sos>`（Start of Sequence）：序列起始符，放在每条句子的开头，告知解码器「从这里开始生成」。
- `<eos>`（End of Sequence）：序列终止符，放在每条句子的末尾，告知模型「句子已结束，停止生成」。

**`map` 函数的作用：**

`dataset.map(tokenize_example, fn_kwargs=fn_kwargs)` 将 `tokenize_example` 函数批量应用到数据集的每一条样本上，并将函数返回的新字段（`en_tokens`、`de_tokens`）追加到原始数据集中，最终返回一个扩充了分词字段的新数据集。相比手写 for 循环，`map` 支持并行加速且内存效率更高。

In [ ]:
max_length = 1_000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)

## 7、运行下方的单元格
重新打印train_data\[0]，验证小写字符串列表以及序列标记的开始/结束符已被成功添加。


In [ ]:
train_data[0]

# 词汇表 Vocabularies

下一个步骤是为源语言和目标语言构建词汇表，将词语映射为数字索引。比如"hello" = 1, "world" = 2, "bye" = 3, "hates" = 4。当向我们的模型提供文本数据时，我们使用词汇表作为look-up-table将字符串转换为标记，然后将标记转换为数字。“hello world”变成了“\[“hello”，“world”]”，然后变成了“\[1,2]”。

In [ ]:
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

en_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["en_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

de_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["de_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

## 8、运行下方两个单元格
验证词汇表，分别打印英语词汇表和德语词汇表的前十个Token。


In [ ]:
en_vocab.get_itos()[:10]

In [ ]:
de_vocab.get_itos()[:10]

## 9、运行下方的单元格
使用get_stoi（stoi = "string to int "）方法获取指定的Token的索引。

In [ ]:
en_vocab["the"]

In [ ]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

In [ ]:
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

词汇表的另一个有用特性是lookup_indices方法。它接受一个Token列表并返回一个索引列表。

## 10、运行下方的单元格
观察从Token列表到索引列表的转换。

In [ ]:
tokens = ["i", "love", "watching", "crime", "shows"]
en_vocab.lookup_indices(tokens)

对应的，lookup_tokens方法使用词汇表将索引列表转换回Token列表。

## 11、运行下方的单元格
观察从索引列表到Token列表的转换。


In [ ]:
en_vocab.lookup_tokens(en_vocab.lookup_indices(tokens))

## 12、添加一个Markdown单元格，在其中解释为什么原本的"crime"被转换成了\<unk>。

**为什么 "crime" 被转换成了 `<unk>`？**

在构建词汇表时设置了 `min_freq=2`，即只有在训练集中出现 **2 次及以上** 的词才会被收录进词汇表。如果 "crime" 在训练语料中的出现次数少于 2 次（或根本未出现），它就不会被加入词汇表。当遇到词汇表中不存在的词时，`set_default_index(unk_index)` 会将其映射为 `<unk>`（未知词）的索引，因此 "crime" 就被替换为了 `<unk>`。

## 13、添加一个Markdown单元格，在其中解释下方两个单元格中代码的作用。


**`numericalize_example` 函数的作用：**

该函数将已分词的 Token 列表转换为对应的整数索引列表，实现从「词」到「数字」的映射：

- `en_vocab.lookup_indices(example["en_tokens"])`：查询英语词汇表，将每个英语 Token 转换为整数索引，得到 `en_ids`；
- `de_vocab.lookup_indices(example["de_tokens"])`：查询德语词汇表，将每个德语 Token 转换为整数索引，得到 `de_ids`；
- 返回包含 `en_ids` 和 `de_ids` 的字典。

**下方 `map` 调用的作用：**

将 `numericalize_example` 批量应用到 train/valid/test 三个数据集的每一条样本，在数据集中追加 `en_ids` 和 `de_ids` 两列整数索引，为后续构建 PyTorch 张量做准备。

In [ ]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

In [ ]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

## 14、运行下方的单元格
重新打印train_data\[0]，验证"en_ids" and "de_ids"被成功添加。


In [ ]:
train_data[0]

Dataset类为我们处理的另一件事是将features转换为正确的类型。每个例子中的索引目前都是基本的Python整数。然而，为了在PyTorch中使用它们，它们需要转换为PyTorch张量。with_format方法将columns参数转换为给定的类型。这里，我们指定类型为“torch”，columns为“en_ids”和“de_ids”（我们想要转换为PyTorch张量的features）。默认情况下，with_format将删除任何不在传递给列的features列表中的features。我们希望保留这些features，这可以通过output_all_columns=True来实现。

In [ ]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

## 15、运行下方的单元格
重新打印train_data[0]，验证“en_ids”和“de_ids”特征被转换为了张量。

In [ ]:
train_data[0]

# Data Loaders

数据准备的最后一步是创建Data Loaders。可以对它们进行迭代以返回一批数据，每一批数据都是一个字典，其中包含数字化的英语和德语句子作为PyTorch张量。

## 16、添加一个Markdown单元格，在其中解释下方两个单元格中的函数的作用。

**`get_collate_fn` 的作用：**

返回一个用于 DataLoader 的 `collate_fn` 函数。由于同一批次（batch）中各句子长度不同，`collate_fn` 会将批次内所有英语句子和德语句子分别用 `pad_sequence` 填充到相同长度（用 `pad_index` 补齐），使之可以拼接成统一形状的张量，供模型批量处理。

**`get_data_loader` 的作用：**

封装 PyTorch 的 `DataLoader`，接收数据集、批次大小、`pad_index` 和是否随机打乱等参数，返回一个可迭代的 DataLoader 对象。每次迭代返回一个包含批量 `en_ids` 和 `de_ids` 张量的字典，供训练/验证/测试循环使用。

In [ ]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [ ]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [ ]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

# 构建模型

我们将分三部分构建模型。编码器，解码器和封装编码器和解码器的seq2seq模型。

# 编码器 Encoder

首先是编码器，它是一个2层的LSTM。

## 17、添加一个Markdown单元格，解释下方单元格中Encoder类的代码。
包括输入参数，核心组件（词嵌入层、LSTM层、Dropout层），forwad函数的处理流程，和输出。

**Encoder 类解析：**

**输入参数：**
- `input_dim`：源语言词汇表大小，即词嵌入层的输入维度。
- `embedding_dim`：词嵌入向量的维度（本实验为 256）。
- `hidden_dim`：LSTM 隐藏状态的维度（本实验为 512）。
- `n_layers`：LSTM 的堆叠层数（本实验为 2）。
- `dropout`：Dropout 概率，用于防止过拟合。

**核心组件：**
- **词嵌入层（Embedding Layer）**：将整数 Token 索引映射为 `embedding_dim` 维的稠密向量。
- **LSTM 层**：多层 LSTM，接收词嵌入序列，逐时间步更新隐藏状态 `hidden` 和细胞状态 `cell`。
- **Dropout 层**：在词嵌入后施加 Dropout，防止模型对特定词过拟合。

**`forward` 处理流程：**
1. 输入 `src`（形状 `[src_len, batch_size]`）经过 Embedding + Dropout，得到 `embedded`（`[src_len, batch_size, embedding_dim]`）；
2. `embedded` 送入 LSTM，得到所有时间步的输出 `outputs` 以及最终的 `hidden`、`cell` 状态；
3. 返回 `(hidden, cell)` 作为上下文向量，传递给解码器作为初始隐藏状态。

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src length, batch size]
        embedded = self.dropout(self.embedding(src))
        # embedded = [src length, batch size, embedding dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs = [src length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # outputs are always from the top hidden layer
        return hidden, cell

# 解码器 Decoder

接下来是解码器，它需要与编码器对齐，同样是一个2层的LSTM。

## 18、添加一个Markdown单元格，描述Decoder的工作流程。

**Decoder 工作流程：**

1. **输入**：每次只接收一个时间步的输入 Token（`input`，形状 `[batch_size]`），以及来自上一时间步（或编码器）的隐藏状态 `hidden` 和细胞状态 `cell`；
2. **维度扩展**：将 `input` 通过 `unsqueeze(0)` 扩展为 `[1, batch_size]`，满足 LSTM 的时序输入格式；
3. **词嵌入 + Dropout**：将输入 Token 映射为嵌入向量，并施加 Dropout；
4. **LSTM 前向传播**：嵌入向量与 `(hidden, cell)` 共同送入 LSTM，输出新的 `output`、`hidden`、`cell`；
5. **线性投影**：通过全连接层 `fc_out` 将 `output`（`[1, batch_size, hidden_dim]`）投影到目标词汇表大小，得到每个词的预测概率分布 `prediction`（`[batch_size, output_dim]`）；
6. **输出**：返回 `(prediction, hidden, cell)`，其中 `prediction` 用于计算损失，`hidden`/`cell` 传入下一时间步。

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # n directions in the decoder will both always be 1, therefore:
        # hidden = [n layers, batch size, hidden dim]
        # context = [n layers, batch size, hidden dim]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, embedding dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output = [seq length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # seq length and n directions will always be 1 in this decoder, therefore:
        # output = [1, batch size, hidden dim]
        # hidden = [n layers, batch size, hidden dim]
        # cell = [n layers, batch size, hidden dim]
        prediction = self.fc_out(output.squeeze(0))
        # prediction = [batch size, output dim]
        return prediction, hidden, cell

# Seq2Seq

## 19、添加一个Markdown单元格，解释下方单元格中Seq2Seq类的代码。
包括forward函数的流程，以及teacher forcing机制。

**Seq2Seq 类解析：**

**`forward` 函数的流程：**
1. 调用 `self.encoder(src)` 对源语言序列编码，获取上下文向量 `(hidden, cell)`；
2. 取目标序列的第一个 Token（`<sos>`）作为解码器的初始输入；
3. 循环 `trg_length - 1` 次，每次调用解码器生成一个预测词及更新后的隐藏/细胞状态；
4. 将每步预测结果存入 `outputs` 张量；
5. 返回完整的 `outputs`，形状为 `[trg_length, batch_size, trg_vocab_size]`。

**Teacher Forcing 机制：**
在训练时，以 `teacher_forcing_ratio` 的概率将**真实目标词**（`trg[t]`）而非模型预测词作为解码器下一步的输入。这种策略可避免早期训练时预测误差的累积，加速模型收敛；但过高的比例会导致模型对真实输入的依赖性过强，推理时性能下降（曝光偏差问题）。

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        # teacher_forcing_ratio is probability to use teacher forcing
        # e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        # last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # first input to the decoder is the <sos> tokens
        input = trg[0, :]
        # input = [batch size]
        for t in range(1, trg_length):
            # insert input token embedding, previous hidden and previous cell states
            # receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, hidden dim]
            # cell = [n layers, batch size, hidden dim]
            # place predictions in a tensor holding predictions for each token
            outputs[t] = output
            # decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio
            # get the highest predicted token from our predictions
            top1 = output.argmax(1)
            # if teacher forcing, use actual next token as next input
            # if not, use predicted token
            input = trg[t] if teacher_force else top1
            # input = [batch size]
        return outputs

# 模型训练

模型初始化

## 20、添加注释
分别将“# 编码器初始化”，“# 解码器初始化”，“# Seq2Seq模型整合”这三行注释加到下方单元格中正确的位置

In [ ]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 编码器初始化
encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

# 解码器初始化
decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

# Seq2Seq模型整合
model = Seq2Seq(encoder, decoder, device).to(device)


权重初始化

In [ ]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

优化器 optimizer

In [ ]:
optimizer = optim.Adam(model.parameters())

损失函数 Loss Function

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

Training Loop:

## 21、给下方单元格中的代码逐行加注释

In [ ]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    model.train()  # 将模型设置为训练模式（启用 Dropout、BatchNorm 等）
    epoch_loss = 0  # 初始化本轮累计损失为 0
    for i, batch in enumerate(data_loader):  # 遍历数据加载器中的每个 batch
        src = batch["de_ids"].to(device)  # 取出德语源序列，并移到指定设备（CPU/GPU）
        trg = batch["en_ids"].to(device)  # 取出英语目标序列，并移到指定设备
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        optimizer.zero_grad()  # 清零上一步的梯度，防止梯度累积
        output = model(src, trg, teacher_forcing_ratio)  # 前向传播，获取模型预测输出
        # output = [trg length, batch size, trg vocab size]
        output_dim = output.shape[-1]  # 获取目标词汇表大小
        output = output[1:].view(-1, output_dim)  # 去除第一个时间步（<sos>），展平为二维张量
        # output = [(trg length - 1) * batch size, trg vocab size]
        trg = trg[1:].view(-1)  # 去除目标序列中的 <sos>，展平为一维张量
        # trg = [(trg length - 1) * batch size]
        loss = criterion(output, trg)  # 计算预测值与真实标签之间的交叉熵损失
        loss.backward()  # 反向传播，计算各参数的梯度
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)  # 梯度裁剪，防止梯度爆炸
        optimizer.step()  # 根据梯度更新模型参数
        epoch_loss += loss.item()  # 累加当前 batch 的损失值
    return epoch_loss / len(data_loader)  # 返回本轮平均损失


Evaluation Loop:

In [ ]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["de_ids"].to(device)
            trg = batch["en_ids"].to(device)
            # src = [src length, batch size]
            # trg = [trg length, batch size]
            output = model(src, trg, 0)  # turn off teacher forcing
            # output = [trg length, batch size, trg vocab size]
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            # output = [(trg length - 1) * batch size, trg vocab size]
            trg = trg[1:].view(-1)
            # trg = [(trg length - 1) * batch size]
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

# 模型训练

In [ ]:
n_epochs = 1 # 因模型训练对计算资源要求较高，此处只设立了一轮训练。
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

# 模型验证

In [ ]:
model.load_state_dict(torch.load("tut1-model.pt"))

In [ ]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        if lower:
            tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [ ]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

In [ ]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

# 22、运行下方单元格，得到测试集第0个索引的翻译
因为epoch只进行了一轮，不会有好的效果的翻译。
感兴趣的同学可自行增加训练轮数，观察loss和翻译质量的变化。

In [ ]:
translation